# Database Fundamentals for AI

This notebook covers the practical database basics you need for local and lightweight AI applications:

- SQLite architecture and storage
- Schema design for AI workloads
- ORM vs raw SQL in SQLite
- File-backed vs in-memory connections
- Async database access with `aiosqlite`

SQLite is a serverless, embedded database engine that normally stores the complete database in a single file on disk, and it also supports in-memory databases for transient execution.


## Learning goals

By the end of this notebook, you should be able to:

1. Explain how SQLite stores data on disk.
2. Design tables for prompt history, token usage, latency, and cache records.
3. Choose between raw SQL and SQLModel for a local AI app.
4. Open file-backed and in-memory SQLite connections correctly.
5. Perform asynchronous CRUD operations with `aiosqlite`.


## 1) Install dependencies

Run this in a notebook cell:

```bash
pip install -U sqlmodel aiosqlite
```

You already get `sqlite3` with Python, so no extra install is needed for the standard library driver.


In [1]:
%pip install -qU sqlmodel aiosqlite


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) SQLite architecture in one minute

SQLite is an in-process, serverless SQL database engine. The official docs describe it as a single-file database that normally stores the complete database in one disk file, with stable and well-defined file format details. 

That makes SQLite a strong fit for:
- local AI prototypes
- desktop apps
- embedded systems
- agent state and lightweight persistence
- small internal tools


## 3) Why SQLite matters for AI workloads

AI apps often need a simple persistence layer for:

- prompt history
- response caches
- token usage logging
- latency metrics
- trace checkpoints
- user/session state

SQLite is a good default when you want low setup cost and a single portable database file. citeturn673490search16turn673490search11


## 4) Storage model

SQLite databases are usually stored in a single main database file. During a transaction, SQLite may also use a rollback journal or a WAL file depending on the journal mode. The architecture doc explains that tables and indexes are implemented with B-trees stored in the same disk file.


In [2]:
from pathlib import Path

db_path = Path("ai_fundamentals.db")
print("Database path:", db_path.resolve())


Database path: D:\personal_docs\course-ai\module-1\db\ai_fundamentals.db


## 5) Schema design for AI workloads

A practical AI schema often includes a few operational tables:

- `prompt_history`: stores prompts and responses
- `llm_latency_log`: stores request timing and model metadata
- `token_usage`: stores input/output token counts
- `cache_entries`: stores reusable answers or lookups

The goal is not to over-normalize too early. For a local AI app, simple transactional tables are often enough.


In [3]:
import sqlite3
from datetime import datetime

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS prompt_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT NOT NULL,
    role TEXT NOT NULL,
    prompt_text TEXT,
    response_text TEXT,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS llm_latency_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_id TEXT NOT NULL,
    model_name TEXT NOT NULL,
    latency_ms REAL NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS token_usage (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_id TEXT NOT NULL,
    model_name TEXT NOT NULL,
    input_tokens INTEGER DEFAULT 0,
    output_tokens INTEGER DEFAULT 0,
    total_tokens INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS cache_entries (
    cache_key TEXT PRIMARY KEY,
    payload TEXT NOT NULL,
    model_name TEXT,
    expires_at TEXT,
    created_at TEXT NOT NULL
)
''')

conn.commit()
conn.close()

print("Tables created in", db_path)


Tables created in ai_fundamentals.db


## 6) Raw SQL vs ORM in SQLite

### Use raw SQL when:
- the query is simple
- you need full control
- you care about minimal abstraction
- you are doing one-off migration or debugging work

### Use SQLModel / SQLAlchemy when:
- the schema is shared across the app
- you want typed models
- you want cleaner CRUD code
- you are building a small-to-medium application with maintainable Python objects

SQLModel is built on Python type annotations and powered by Pydantic and SQLAlchemy. The official tutorial shows how to define table models and work with sessions step by step.


In [4]:
from typing import Optional
from sqlmodel import SQLModel, Field, create_engine, Session, select

class PromptHistory(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    session_id: str
    role: str
    prompt_text: Optional[str] = None
    response_text: Optional[str] = None
    created_at: str

class TokenUsage(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    request_id: str
    model_name: str
    input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    created_at: str

engine = create_engine(f"sqlite:///{db_path}", echo=False)
SQLModel.metadata.create_all(engine)

print("SQLModel tables created.")


SQLModel tables created.


In [5]:
# Insert a sample row with SQLModel
now = datetime.utcnow().isoformat()

with Session(engine) as session:
    row = PromptHistory(
        session_id="sess-001",
        role="user",
        prompt_text="Summarize the meeting notes.",
        response_text="Summary of meeting notes...",
        created_at=now,
    )
    session.add(row)
    session.commit()
    session.refresh(row)
    print("Inserted row:", row)


Inserted row: response_text='Summary of meeting notes...' session_id='sess-001' role='user' prompt_text='Summarize the meeting notes.' id=1 created_at='2026-06-08T10:37:30.594134'


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_916\3415703439.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()


In [6]:
# Read data back with SQLModel
with Session(engine) as session:
    statement = select(PromptHistory).where(PromptHistory.session_id == "sess-001")
    rows = session.exec(statement).all()
    for r in rows:
        print(r)


response_text='Summary of meeting notes...' session_id='sess-001' role='user' prompt_text='Summarize the meeting notes.' id=1 created_at='2026-06-08T10:37:30.594134'


## 7) When raw SQL is still the right choice

Raw SQL is often better for:
- fast inserts
- migrations
- schema inspection
- custom aggregations
- debugging query behavior

For AI logging tables, a raw SQL insert is often perfectly fine when the write path is simple and performance matters.


In [7]:
# Raw SQL example for logging model latency
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
INSERT INTO llm_latency_log (
    request_id, model_name, latency_ms, status, created_at
) VALUES (?, ?, ?, ?, ?)
''', ("req-001", "llama-3.3-70b-versatile", 842.5, "ok", datetime.utcnow().isoformat()))

conn.commit()

cur.execute("SELECT request_id, model_name, latency_ms, status FROM llm_latency_log")
print(cur.fetchall())

conn.close()


[('req-001', 'llama-3.3-70b-versatile', 842.5, 'ok')]


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_916\3637701783.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ''', ("req-001", "llama-3.3-70b-versatile", 842.5, "ok", datetime.utcnow().isoformat()))


## 8) File-backed vs in-memory connections

### File-backed database
Use a file path like:

```python
sqlite3.connect("ai_fundamentals.db")
```

This is best for:
- persistent local storage
- repeatable development work
- storing state between runs

### In-memory database
Use:

```python
sqlite3.connect(":memory:")
```

This is best for:
- tests
- disposable experiments
- temporary data
- fast isolated demos

SQLite documents in-memory databases separately from disk-backed databases. citeturn673490search20


In [8]:
# File-backed connection
file_conn = sqlite3.connect("ai_fundamentals.db")
print("File-backed DB connected:", file_conn is not None)
file_conn.close()

# In-memory connection
mem_conn = sqlite3.connect(":memory:")
mem_cur = mem_conn.cursor()
mem_cur.execute("CREATE TABLE demo (id INTEGER PRIMARY KEY, name TEXT)")
mem_cur.execute("INSERT INTO demo (name) VALUES (?)", ("temporary",))
mem_cur.execute("SELECT * FROM demo")
print("In-memory rows:", mem_cur.fetchall())
mem_conn.close()


File-backed DB connected: True
In-memory rows: [(1, 'temporary')]


## 9) Async database access with aiosqlite

`aiosqlite` provides an async interface to SQLite and mirrors the standard `sqlite3` API with async connection and cursor methods. It is useful when your app already uses `asyncio` and you do not want blocking database calls to stall the event loop.


In [10]:
import aiosqlite

async def init_async_db():
    async with aiosqlite.connect(db_path) as db:
        await db.execute("""
        CREATE TABLE IF NOT EXISTS async_cache_entries (
            cache_key TEXT PRIMARY KEY,
            payload TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
        """)
        await db.commit()

await init_async_db()

print("Async table created.")

Async table created.


In [11]:
from datetime import datetime

async def async_create_read_update_delete():
    async with aiosqlite.connect(db_path) as db:

        # Create
        await db.execute(
            """
            INSERT OR REPLACE INTO async_cache_entries
            (cache_key, payload, created_at)
            VALUES (?, ?, ?)
            """,
            (
                "prompt:hello",
                "cached response",
                datetime.utcnow().isoformat(),
            ),
        )
        await db.commit()

        # Read
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM async_cache_entries
            WHERE cache_key = ?
            """,
            ("prompt:hello",),
        ) as cursor:
            rows = await cursor.fetchall()
            print("Read:", rows)

        # Update
        await db.execute(
            """
            UPDATE async_cache_entries
            SET payload = ?
            WHERE cache_key = ?
            """,
            (
                "updated cached response",
                "prompt:hello",
            ),
        )
        await db.commit()

        # Verify update
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM async_cache_entries
            WHERE cache_key = ?
            """,
            ("prompt:hello",),
        ) as cursor:
            print("After update:", await cursor.fetchall())

        # Delete
        await db.execute(
            """
            DELETE FROM async_cache_entries
            WHERE cache_key = ?
            """,
            ("prompt:hello",),
        )
        await db.commit()

        # Verify delete
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM async_cache_entries
            WHERE cache_key = ?
            """,
            ("prompt:hello",),
        ) as cursor:
            print("After delete:", await cursor.fetchall())


await async_create_read_update_delete()

Read: [('prompt:hello', 'cached response')]
After update: [('prompt:hello', 'updated cached response')]
After delete: []


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_916\3838460262.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(),


## 10) Choosing between sqlite3 and aiosqlite

### Use `sqlite3` when:
- your script is synchronous
- you want the simplest possible setup
- your data access happens in small batches

### Use `aiosqlite` when:
- your app is async already
- you want non-blocking DB access
- you are building an agent, API server, or event-driven workflow


## 11) AI workload schema ideas

A simple but effective AI schema could be:

- `prompt_history(session_id, role, prompt_text, response_text, created_at)`
- `llm_latency_log(request_id, model_name, latency_ms, status, created_at)`
- `token_usage(request_id, model_name, input_tokens, output_tokens, total_tokens, created_at)`
- `cache_entries(cache_key, payload, model_name, expires_at, created_at)`

This covers:
- auditability
- performance monitoring
- cost tracking
- cache reuse
- debugging


## 12) Practical recommendations

- Use a single SQLite file for local development and lightweight production.
- Use raw SQL for simple write paths and debugging.
- Use SQLModel when you want typed models and cleaner CRUD code.
- Use `:memory:` for tests.
- Use `aiosqlite` when your application is async.
- Keep schema small, explicit, and focused on the AI workflow.


## 13) Mini exercise

Try this next:

1. Add a `conversation_id` column to `prompt_history`.
2. Add a `response_time_ms` column to `prompt_history`.
3. Create an index for `session_id`.
4. Store a new row every time your LLM returns a response.
5. Query the last 10 interactions for a given session.


## Key takeaways

- SQLite is a single-file, serverless, embedded database engine.
- AI apps often need prompt history, token tracking, latency logs, and cache tables.
- Raw SQL is best for simple direct control.
- SQLModel is better when you want typed models and maintainable CRUD.
- File-backed databases persist data; in-memory databases are temporary.
- `aiosqlite` gives you async, non-blocking SQLite access in Python. 


## References

- SQLite architecture: https://sqlite.org/arch.html
- SQLite file format: https://sqlite.org/fileformat.html
- SQLite single-file database: https://www.sqlite.org/onefile.html
- SQLite documentation: https://sqlite.org/docs.html
- SQLite appropriate uses: https://sqlite.org/whentouse.html
- SQLModel tutorial: https://sqlmodel.tiangolo.com/tutorial/
- SQLModel homepage: https://sqlmodel.tiangolo.com/
- aiosqlite: https://github.com/omnilib/aiosqlite
- aiosqlite docs: https://aiosqlite.omnilib.dev/en/latest/
- Intro to SQLite in Python: https://www.geeksforgeeks.org/python/introduction-to-sqlite-in-python/
- SQLite tutorial: https://www.geeksforgeeks.org/sqlite/sqlite-tutorial/
